In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
just_checking_integrity=False

In [ ]:
import gc
import pandas as pd
import scipy.sparse as sps

In [ ]:
valid=sps.load_npz("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/processed_data/URM_valid_bought.npz")
bought=valid.indices

In [ ]:
dataset=None
dataset_max=None
#["UCF","ICBF","ICF","Graph","TopPop100","TopPop50","TopPop15"]
#for file in ["UCF","ICBF","Graph","ICF","UCF_W","ICBF_W","Graph_W","ICF_W","UCF_WW","ICBF_WW","Graph_WW","ICF_WW"]:
for file in ["UCF_W","ICBF_W","Graph_W","ICF_W"]:
    print(gc.collect())
    print(f"loading {file}")
    df=pd.read_csv(f"/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/traditional_recs/train/{file}.csv")
    print(len(df))
    print(f"loaded {file}")
    for col in df:
        if "Max" in col:
            df_max=df[["Session_Id",col]].drop_duplicates()
        elif "Score" in col:
            df_feats=df[["Session_Id","Item_ID",col]]
    if dataset is None:
        dataset=df_feats
        dataset_max=df_max
    else:
        dataset=dataset.merge(df_feats,on=["Session_Id","Item_ID"],how="outer")
        dataset_max=dataset_max.merge(df_max,on=["Session_Id"],how="outer")
    del df


del dataset_max
del df_feats
del df_max

0
loading UCF_W
18588455
loaded UCF_W
0
loading ICBF_W
16931962
loaded ICBF_W
0
loading Graph_W
9381571
loaded Graph_W
0
loading ICF_W
20341641
loaded ICF_W


In [ ]:
total_sessions = len(dataset['Session_Id'].unique())
print("Total sessions:", total_sessions)

Total sessions: 81618


In [ ]:
dataset

,Session_Id,Item_ID,Score_UCF_W,Score_ICBF_W,Score_Graph_W,Score_ICF_W
0,918382,78,0.515642,NaN,0.377850,0.587459
1,918382,178,0.281229,NaN,0.142546,0.412063
2,918382,230,0.127506,NaN,0.055109,0.022295
3,918382,242,0.395681,0.775898,0.273503,0.514097
4,918382,292,0.200113,0.601746,0.105606,0.458708
...,...,...,...,...,...,...
35891117,999999,14138,NaN,NaN,NaN,0.505847
35891118,999999,16357,NaN,NaN,NaN,0.005289
35891119,999999,16479,NaN,NaN,NaN,0.541090
35891120,999999,16520,NaN,NaN,NaN,0.359696


In [ ]:
import ast
submission_df=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/NN/GRU/train/candidates.csv")
if just_checking_integrity:
    submission_df["score"]=submission_df["score"].apply(lambda x:x.replace("-inf","0")).apply(lambda x:x.replace("+inf","0")).apply(lambda x:x.replace("inf","0"))
    submission_df["score"]=submission_df["score"].map(ast.literal_eval)
else:
    submission_df["score"]=submission_df["score"].map(ast.literal_eval)
submission_df["code"]=submission_df["code"].map(ast.literal_eval)
submission_df["rank"]=submission_df["rank"].map(ast.literal_eval)
submission_df = submission_df.explode(['code','score', 'rank'])
submission_df[["session_id","code","rank"]]=submission_df[["session_id","code","rank"]].astype(int)
submission_df.rename(columns={"code":"Item_ID","session_id":"Session_Id","rank":"rank_gru","score":"score_gru"},inplace=True)

In [ ]:
submission_df.head()

,Session_Id,Item_ID,score_gru,rank_gru
0,918382,3844,0.021967,1
0,918382,3678,0.016598,2
0,918382,1189,0.015711,3
0,918382,446,0.01126,4
0,918382,1813,0.009247,5


In [ ]:
# import ast
# submission_df_tr=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/NN/Transformer/train/candidates.csv")
# if just_checking_integrity:
#     submission_df_tr["score"]=submission_df_tr["score"].apply(lambda x:x.replace("-inf","0"))
#     submission_df_tr["score"]=submission_df_tr["score"].map(ast.literal_eval)
# else:
#     submission_df_tr["score"]=submission_df_tr["score"].map(ast.literal_eval)
# submission_df_tr["code"]=submission_df_tr["code"].map(ast.literal_eval)
# submission_df_tr["rank"]=submission_df_tr["rank"].map(ast.literal_eval)
# submission_df_tr = submission_df_tr.explode(['code','score', 'rank'])
# submission_df_tr[["session_id","code","rank"]]=submission_df_tr[["session_id","code","rank"]].astype(int)
# submission_df_tr.rename(columns={"code":"Item_ID","session_id":"Session_Id","rank":"rank_transformer","score":"score_transformer"},inplace=True)

In [ ]:
# import ast
# submission_df_LSTM=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/candidates/NN/LSTM/train/candidates.csv")
# if just_checking_integrity:
#     submission_df_LSTM["score"]=submission_df_LSTM["score"].apply(lambda x:x.replace("-inf","0"))
#     submission_df_LSTM["score"]=submission_df_LSTM["score"].map(ast.literal_eval)
# else:
#     submission_df_LSTM["score"]=submission_df_LSTM["score"].map(ast.literal_eval)
# submission_df_LSTM["code"]=submission_df_LSTM["code"].map(ast.literal_eval)
# submission_df_LSTM["rank"]=submission_df_LSTM["rank"].map(ast.literal_eval)
# submission_df_LSTM = submission_df_LSTM.explode(['code','score', 'rank'])
# submission_df_LSTM[["session_id","code","rank"]]=submission_df_LSTM[["session_id","code","rank"]].astype(int)
# submission_df_LSTM.rename(columns={"code":"Item_ID","session_id":"Session_Id","rank":"rank_LSTM","score":"score_LSTM"},inplace=True)

In [ ]:
dataset=dataset.merge(submission_df,on=["Session_Id","Item_ID"],how="outer")
#dataset=dataset.merge(submission_df_tr,on=["Session_Id","Item_ID"],how="outer")
#dataset=dataset.merge(submission_df_LSTM,on=["Session_Id","Item_ID"],how="outer")

# dataset=dataset.merge(dataset_max,on=["Session_Id"],how="left")


In [ ]:
dataset

,Session_Id,Item_ID,Score_UCF_W,Score_ICBF_W,Score_Graph_W,Score_ICF_W,score_gru,rank_gru
0,918382,78,0.515642,NaN,0.377850,0.587459,NaN,NaN
1,918382,178,0.281229,NaN,0.142546,0.412063,NaN,NaN
2,918382,230,0.127506,NaN,0.055109,0.022295,NaN,NaN
3,918382,242,0.395681,0.775898,0.273503,0.514097,0.003217,14.0
4,918382,292,0.200113,0.601746,0.105606,0.458708,NaN,NaN
...,...,...,...,...,...,...,...,...
39064205,999999,2019,NaN,NaN,NaN,NaN,0.000588,92.0
39064206,999999,2316,NaN,NaN,NaN,NaN,0.000588,93.0
39064207,999999,3320,NaN,NaN,NaN,NaN,0.000578,96.0
39064208,999999,968,NaN,NaN,NaN,NaN,0.000561,99.0


In [ ]:
if just_checking_integrity:
    dataset=dataset[:int(0.5*len(dataset))]

In [ ]:
# del dataset_max
del submission_df

In [ ]:
import numpy as np
sessions=np.unique(dataset["Session_Id"])
train=pd.DataFrame()
train["Session_Id"]=sessions
if just_checking_integrity:
    train["Item_ID"]=bought[:len(sessions)]
else:
    train["Item_ID"]=bought
train["target"]=1

In [ ]:
gc.collect()

242

In [ ]:
dataset=dataset.merge(train,on=["Session_Id","Item_ID"],how="left")
dataset.fillna(0,inplace=True)
gc.collect()

0

In [ ]:
dataset["target"]=dataset["target"].astype("int8")
gc.collect()

0

In [ ]:
sessions_containing_bought_among_candidates=dataset[["Session_Id","target"]].groupby("Session_Id").agg(sum).reset_index()
sessions_containing_bought_among_candidates=sessions_containing_bought_among_candidates.loc[sessions_containing_bought_among_candidates["target"]>0]

dataset=dataset[dataset['Session_Id'].isin(sessions_containing_bought_among_candidates["Session_Id"].tolist())]

In [ ]:
NHITS=len(dataset['Session_Id'].unique())
HITRATE=NHITS/81600
HITRATE

0.7326470588235294

In [ ]:
dataset

,Session_Id,Item_ID,Score_UCF_W,Score_ICBF_W,Score_Graph_W,Score_ICF_W,score_gru,rank_gru,target
209,918383,55,0.054177,0.632362,0.005355,0.018442,0.000000,0.0,0
210,918383,122,0.025767,0.000000,0.000000,0.000000,0.000000,0.0,0
211,918383,189,0.129413,0.000000,0.020602,0.278846,0.000446,76.0,0
212,918383,215,0.036236,0.000000,0.000000,0.000000,0.000000,0.0,0
213,918383,260,0.039900,0.000000,0.003595,0.012055,0.000000,0.0,0
...,...,...,...,...,...,...,...,...,...
39064205,999999,2019,0.000000,0.000000,0.000000,0.000000,0.000588,92.0,0
39064206,999999,2316,0.000000,0.000000,0.000000,0.000000,0.000588,93.0,0
39064207,999999,3320,0.000000,0.000000,0.000000,0.000000,0.000578,96.0,0
39064208,999999,968,0.000000,0.000000,0.000000,0.000000,0.000561,99.0,0


In [ ]:
dataset_macro_features = pd.read_csv('/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/processed_data/macro_feats_NN.csv',
                                     usecols=['session_id',
                                                'session_similarity',
                                                'session_similarity_uniques', 'session_similarity_feats', 'session_similarity_feats_uniques',
                                              'length_of_session_seconds',
                   'avg_time_spent_per_item_seconds',
                   'variance_time_spent_per_item_seconds',
                   'n_seen_items',
                   'n_unique_items',])

dataset_macro_features.rename(columns={'session_id':'Session_Id'}, inplace=True)
dataset = dataset.merge(dataset_macro_features, on='Session_Id', how='left')
del dataset_macro_features

In [ ]:
#dataset.drop('Max_Score_TopPop15', axis=1, inplace=True)

In [ ]:
item_features=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/original_data/item_features.csv")
subgroup_cat_feat=pd.read_csv("/content/drive/MyDrive/RS2024/Trial02/recsys2022/dataset/original_data/subgroup_cat_feat.csv")

In [ ]:
subgroup_tuples = set([tuple(x) for x in subgroup_cat_feat.values])
dataset['weight'] = 1.0
purchased_items = dataset[dataset['target'] == 1]
alpha = 0.5

# For each purchased item, check if its features match the subgroup
for index, row in purchased_items.iterrows():
    item_id = row['Item_ID']
    cat_feat = item_features[item_features['item_id'] == item_id][['feature_category_id','feature_value_id']]

    item_feature_tuples = set([tuple(x) for x in cat_feat.values])

    if item_feature_tuples & subgroup_tuples: # Check if there is at least 1 intersection between the sets, if so assign higher weight to all rows in the session
        session_id = row['Session_Id']
        dataset.loc[dataset['Session_Id'] == session_id, 'weight'] = 1+alpha

In [ ]:
def calculate_mrr(data, k=100):
    """
    Calculate MRR@k for given data.
    """
    mrr_score = 0.0
    n_sessions = data['Session_Id'].nunique()

    for session_id, group in data.groupby("Session_Id"):
        sorted_group = group.sort_values('predicted_score', ascending=False).head(k)
        sorted_group = sorted_group.reset_index(drop=True)                      # Reset index to reflect the new order
        correct_answer_index = sorted_group[sorted_group['target'] == 1].index  # Find the index (rank) of the correct answer within the sorted group
        if not correct_answer_index.empty:
            rank = correct_answer_index[0] + 1
            mrr_score += 1.0 / rank

    return mrr_score / n_sessions if n_sessions > 0 else 0

In [ ]:
from sklearn.model_selection import KFold
import lightgbm
import os

k=10
score=0
kf = KFold(n_splits=k)
mrr_score_subgroup = 0

for train_index, valid_index in kf.split(train):

    train_sessions=train["Session_Id"][train_index].tolist()
    valid_sessions=train["Session_Id"][valid_index].tolist()

    train_set=dataset[dataset["Session_Id"].isin(train_sessions)]
    valid_set=dataset[dataset["Session_Id"].isin(valid_sessions)]

    train_set=train_set.sort_values("Session_Id")
    valid_set=valid_set.sort_values("Session_Id")

    qids_train = train_set[["Session_Id"]].groupby("Session_Id")["Session_Id"].count().to_numpy()
    qids_valid = valid_set[["Session_Id"]].groupby("Session_Id")["Session_Id"].count().to_numpy()

    X_train=train_set.drop(["target","Session_Id","Item_ID"],axis=1)
    y_train=train_set["target"]
    sample_weight = train_set["weight"].values  # Extracting sample weights for the training set


    X_valid=valid_set.drop(["target","Session_Id","Item_ID"],axis=1)
    y_valid=valid_set["target"]

    model = lightgbm.LGBMRanker(
    objective="lambdarank",
    #objective="rank_xendcg",
    #metric="mrr",
    # n_estimators=2000,
    n_estimators=100,
    lambdarank_truncation_level=103,
    reg_alpha=1e-4,
    learning_rate=0.05,
    min_child_samples=400,
    subsample=0.8,
    subsample_freq=20
    )
    ranker=model.fit(
        X=X_train,
        y=y_train,
        group=qids_train,
        sample_weight=sample_weight,  # Use sample weights
        eval_set=[(X_valid, y_valid)],
        eval_group=[qids_valid],
        eval_at=(100),
        callbacks=[lightgbm.log_evaluation(),lightgbm.early_stopping(20,first_metric_only=True)],
        eval_metric=["map","ndcg"]
    )
    ranker.booster_.save_model(f'/content/drive/MyDrive/RS2024/Trial02/recsys2022/rankers/Innovation_model_{index}.txt')

    # Predict on validation set
    valid_set['predicted_score'] = ranker.predict(X_valid)
    valid_set_sorted = valid_set.sort_values(['Session_Id', 'predicted_score'], ascending=[True, False])

    # Calculate MRR@100 for all sessions
    # score+=np.max(ranker._evals_result["valid_0"]["map@100"])
    score += calculate_mrr(valid_set_sorted, k=100)

    # Calculate MRR@100 for the subgroup
    valid_set_subgroup = valid_set_sorted[valid_set_sorted['Session_Id'].isin(valid_set[valid_set['weight'] > 1.0]['Session_Id'])]

    if not valid_set_subgroup.empty:
        mrr_score_subgroup += calculate_mrr(valid_set_subgroup, k=100)


[LightGBM] [Info] Calculating query weights...
[LightGBM] [Info] Total groups: 53705, total data: 25402180
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.527713 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3318
[LightGBM] [Info] Number of data points in the train set: 25402180, number of used features: 16
[LightGBM] [Info] Total groups: 6079, total data: 2854352
[1]	valid_0's map@100: 0.190966	valid_0's ndcg@100: 0.306154
Training until validation scores don't improve for 20 rounds
[2]	valid_0's map@100: 0.221755	valid_0's ndcg@100: 0.331682
[3]	valid_0's map@100: 0.229012	valid_0's ndcg@100: 0.338128
[4]	valid_0's map@100: 0.229624	valid_0's ndcg@100: 0.338721
[5]	valid_0's map@100: 0.229505	valid_0's ndcg@100: 0.338922
[6]	valid_0's map@100: 0.231591	valid_0's ndcg@100: 0.34051
[7]	valid_0's map@100: 0.231631	valid_0's ndcg@100: 0

In [ ]:
final_performance = score / k
final_performance_subgroup = mrr_score_subgroup / k
print("Overall MRR@100 with label in candidates = ", final_performance)
print("Subgroup MRR@100 with label in candidates = ", final_performance_subgroup)

Overall MRR@100 with label in candidates =  0.2550750856206885
Subgroup MRR@100 with label in candidates =  0.29816309968649324


In [ ]:
print("Normalized score Overall MRR@100 ", final_performance*HITRATE)
print("Normalized score Subgroup MRR@100 ", final_performance_subgroup*HITRATE)

Normalized score Overall MRR@100  0.18688001125915735
Normalized score Subgroup MRR@100  0.21844831803501608
